# LangChain Agents Workshop

## Building AI Agents with LangChain

This notebook covers:

**PART 1: Core Agent Building** (Following the deck)
- Setup & Configuration
- Chains vs Agents - Understanding the difference
- Creating Tools
- Building Your First Agent
- Agents with Memory
- Debugging Agent Behavior

**PART 2: Extended Coverage** (Advanced Topics)
- Real Web Search with Tavily
- Agent Types Deep Dive
- Custom Tools with Pydantic Schemas
- Advanced Guardrails
- Practical Project: Research Assistant

---

# PART 1: Core Agent Building

---

## Section 1.1: Setup & Configuration

In [ ]:
# Install required dependencies
!pip install -q \
    langchain>=0.3.0 \
    langchain-openai>=0.2.0 \
    langchain-community>=0.3.0 \
    langchain-core>=0.3.0 \
    langchainhub>=0.1.0 \
    tavily-python>=0.3.0 \
    python-dotenv>=1.0 \
    requests>=2.31.0 \
    pydantic>=2.0

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ============================================================
# CONFIGURATION: Choose ONE of the options below
# ============================================================

# OPTION 1: OpenRouter (supports multiple models - GPT, Claude, etc.)
USE_OPENROUTER = True  # Set to False for direct OpenAI

if USE_OPENROUTER:
    API_KEY = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
    BASE_URL = "https://openrouter.ai/api/v1"
    MODEL_NAME = "openai/gpt-4o-mini"  # or "anthropic/claude-3.5-sonnet"
    print("Using OpenRouter API")
else:
    # OPTION 2: Direct OpenAI
    API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
    BASE_URL = None  # Uses default OpenAI endpoint
    MODEL_NAME = "gpt-4o-mini"
    print("Using OpenAI API directly")

# Tavily API Key for web search (Part 2)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "your-tavily-api-key")

print("Configuration loaded successfully!")

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the LLM
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,
    temperature=0,  # Lower temperature for more consistent tool selection
    max_tokens=1000,
)

# Test the connection
response = llm.invoke("Say 'Hello, Agents!' in a creative way.")
print(response.content)

## Section 1.2: Chains vs Agents - The Key Difference

Let's first understand why we need agents by looking at what chains can and cannot do.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# A simple chain - FIXED sequence
# Step 1: Analyze topic -> Step 2: Generate summary

analyze_prompt = ChatPromptTemplate.from_template(
    "Identify 3 key points about: {topic}"
)

summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize these points in one sentence: {points}"
)

# Chain them together
chain = (
    {"points": analyze_prompt | llm | StrOutputParser()}
    | summarize_prompt 
    | llm 
    | StrOutputParser()
)

result = chain.invoke({"topic": "artificial intelligence"})
print("Chain Result:")
print(result)

In [ ]:
# The LIMITATION of chains:
# What if we need to do a calculation that requires external tools?
# Or search for real-time information?

# Chain can only use what's built into it - it can't dynamically decide
# "Hey, I need to look this up" or "I should calculate this"

# Example: Ask the chain to do math - it might hallucinate!
math_prompt = ChatPromptTemplate.from_template(
    "What is 7823 * 456? Just give me the number."
)

math_chain = math_prompt | llm | StrOutputParser()
result = math_chain.invoke({})
print(f"Chain's answer: {result}")
print(f"Actual answer: {7823 * 456}")
print("\n-> LLMs can make math errors. Agents can use a calculator tool!")

### Key Insight

| Chains | Agents |
|--------|--------|
| Fixed sequence: Step 1 → Step 2 → Step 3 | Dynamic: LLM decides what to do |
| You define the path | Model finds the path |
| Good for predictable tasks | Good for open-ended tasks |

**Agents** can use **tools** to extend their capabilities - calculators, web search, APIs, databases, etc.

## Section 1.3: Creating Your First Tools

Tools are functions that agents can call. Let's create two tools:
1. **Calculator** - for math operations
2. **Weather** - using free Open-Meteo API

In [ ]:
from langchain_core.tools import tool
import requests

# ============================================================
# TOOL 1: Calculator
# ============================================================

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.
    
    Use this tool when you need to perform calculations like:
    - Basic arithmetic: addition, subtraction, multiplication, division
    - Complex expressions: (2 + 3) * 4 / 2
    - Temperature conversions: (celsius * 9/5) + 32
    
    Args:
        expression: A valid Python math expression as a string
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # Using eval with limited scope for safety
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names, {})
        return str(result)
    except Exception as e:
        return f"Error calculating: {e}"

# Test the calculator
print("Calculator test:")
print(f"2 + 2 = {calculator.invoke('2 + 2')}")
print(f"(15 * 9/5) + 32 = {calculator.invoke('(15 * 9/5) + 32')}")

In [ ]:
# ============================================================
# TOOL 2: Weather (using free Open-Meteo API - no key needed!)
# ============================================================

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Use this tool when you need current weather information for a location.
    Returns temperature in Celsius and weather condition.
    
    Args:
        city: The name of the city (e.g., 'Tokyo', 'London', 'New York')
    
    Returns:
        Current temperature and weather condition
    """
    try:
        # Step 1: Get coordinates for the city using geocoding
        geocode_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_response = requests.get(geocode_url, timeout=10)
        geo_data = geo_response.json()
        
        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return f"Could not find location: {city}"
        
        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        location_name = geo_data["results"][0]["name"]
        country = geo_data["results"][0].get("country", "")
        
        # Step 2: Get weather data
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,weather_code"
        weather_response = requests.get(weather_url, timeout=10)
        weather_data = weather_response.json()
        
        temp = weather_data["current"]["temperature_2m"]
        weather_code = weather_data["current"]["weather_code"]
        
        # Map weather codes to conditions
        weather_conditions = {
            0: "Clear sky",
            1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Foggy", 48: "Depositing rime fog",
            51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
            61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
            71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
            80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
            95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail"
        }
        condition = weather_conditions.get(weather_code, "Unknown")
        
        return f"Weather in {location_name}, {country}: {temp}°C, {condition}"
    
    except Exception as e:
        return f"Error getting weather: {e}"

# Test the weather tool
print("Weather tool test:")
print(get_weather.invoke("Tokyo"))
print(get_weather.invoke("London"))

In [ ]:
# Let's examine what a tool looks like to the LLM
print("=== Tool Structure ===")
print(f"\nName: {calculator.name}")
print(f"Description: {calculator.description}")
print(f"\nArgs Schema: {calculator.args_schema.schema()}")

print("\n" + "="*50)

print(f"\nName: {get_weather.name}")
print(f"Description: {get_weather.description}")

## Section 1.4: Building Your First Agent

Now let's build an agent following the **6 steps from the deck**:

1. Import the model ✅ (already done)
2. Create tools ✅ (just did this)
3. Choose agent type
4. Initialize AgentExecutor
5. Run sample queries
6. Turn on verbose logs

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Step 2: Collect our tools
tools = [calculator, get_weather]

# Step 3: Choose agent type - we'll use Tool Calling Agent
# This works with models that support function/tool calling (GPT-4, Claude, etc.)

# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant with access to tools.
When asked about weather, use the get_weather tool.
When asked to calculate something, use the calculator tool.
Always show your work and explain your reasoning."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # This is where the agent's work goes
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)

print("Agent created successfully!")

In [ ]:
# Step 4: Initialize AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Step 6: Turn on verbose logs!
    max_iterations=5,  # Safety limit
    handle_parsing_errors=True
)

print("AgentExecutor ready!")

In [ ]:
# Step 5: Run sample queries

# Query 1: Simple calculation
print("=" * 60)
print("QUERY 1: Simple calculation")
print("=" * 60)

result = agent_executor.invoke({"input": "What is 7823 multiplied by 456?"})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Query 2: Weather lookup
print("=" * 60)
print("QUERY 2: Weather lookup")
print("=" * 60)

result = agent_executor.invoke({"input": "What's the weather like in Paris right now?"})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Query 3: Multi-step task (requires multiple tools!)
print("=" * 60)
print("QUERY 3: Multi-step task")
print("=" * 60)

result = agent_executor.invoke({
    "input": "What's the weather in Tokyo? And convert the temperature to Fahrenheit."
})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Query 4: Task that doesn't need tools
print("=" * 60)
print("QUERY 4: No tools needed")
print("=" * 60)

result = agent_executor.invoke({"input": "What is the capital of France?"})
print(f"\nFinal Answer: {result['output']}")

### Understanding the Verbose Output

When `verbose=True`, you see the agent's reasoning:

```
> Entering new AgentExecutor chain...
Invoking: `calculator` with `{'expression': '7823 * 456'}`   <- Tool called
3567288                                                        <- Tool output
The result is 3,567,288.                                       <- Final answer
> Finished chain.
```

This is the **agent loop** in action:
1. Think: "I need to calculate this"
2. Act: Call calculator tool
3. Observe: Get result
4. Answer: Formulate response

## Section 1.5: Agents with Memory

Currently, each query is independent. Let's add memory so the agent remembers previous interactions.

In [ ]:
from langchain.memory import ConversationBufferMemory

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Update prompt to include chat history
prompt_with_memory = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant with access to tools.
You can use get_weather to check weather and calculator for math.
Remember previous conversation context when answering follow-up questions."""),
    MessagesPlaceholder(variable_name="chat_history"),  # Memory goes here!
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Create new agent with memory prompt
agent_with_memory = create_tool_calling_agent(llm, tools, prompt_with_memory)

# Create executor with memory
agent_executor_with_memory = AgentExecutor(
    agent=agent_with_memory,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5
)

print("Agent with memory created!")

In [ ]:
# Conversation with memory - Turn 1
print("=" * 60)
print("TURN 1")
print("=" * 60)

result1 = agent_executor_with_memory.invoke({"input": "What's the weather in New York?"})
print(f"\nAnswer: {result1['output']}")

In [ ]:
# Turn 2 - Follow-up question (refers to previous context)
print("=" * 60)
print("TURN 2 - Follow-up")
print("=" * 60)

result2 = agent_executor_with_memory.invoke({"input": "Convert that temperature to Fahrenheit"})
print(f"\nAnswer: {result2['output']}")

In [ ]:
# Turn 3 - Another follow-up
print("=" * 60)
print("TURN 3 - More context")
print("=" * 60)

result3 = agent_executor_with_memory.invoke({
    "input": "Now check the weather in London and tell me which city is warmer"
})
print(f"\nAnswer: {result3['output']}")

In [ ]:
# View the memory contents
print("\n" + "=" * 60)
print("MEMORY CONTENTS")
print("=" * 60)

print(memory.load_memory_variables({}))

## Section 1.6: Debugging Agent Behavior

Understanding how to debug when things go wrong.

In [ ]:
# Example: Ambiguous query - watch how the agent reasons
print("=" * 60)
print("DEBUGGING EXAMPLE: Ambiguous query")
print("=" * 60)

result = agent_executor.invoke({
    "input": "Compare 5 cities and tell me about them"
})
print(f"\nAnswer: {result['output']}")

In [ ]:
# Demonstrating a poorly described tool (what NOT to do)

@tool
def mystery_tool(x: str) -> str:
    """Does something."""  # BAD description!
    return f"Processed: {x}"

# The agent won't know when to use this tool!
print("Bad tool description example:")
print(f"Name: {mystery_tool.name}")
print(f"Description: {mystery_tool.description}")
print("\n-> The agent has no idea when to use this tool!")

In [ ]:
# Better version
@tool
def format_for_display(text: str) -> str:
    """Format text for display by adding decorative borders.
    
    Use this when you need to make text visually stand out,
    such as for titles, announcements, or important messages.
    
    Args:
        text: The text to format with decorative borders
    
    Returns:
        The text wrapped in decorative ASCII borders
    """
    border = "=" * (len(text) + 4)
    return f"{border}\n| {text} |\n{border}"

print("Good tool description example:")
print(f"Name: {format_for_display.name}")
print(f"Description: {format_for_display.description}")
print("\n-> Now the agent knows exactly when to use it!")

### Debugging Checklist

1. **Enable verbose=True** - Always your first step
2. **Check tool descriptions** - Are they clear and specific?
3. **Verify tool outputs** - Is the format what the model expects?
4. **Watch for loops** - Use max_iterations to prevent runaway costs
5. **Test with simple queries** - Isolate the problem

---

# PART 2: Extended Coverage

Advanced topics beyond the basic deck content.

---

## Section 2.1: Real Web Search with Tavily

Tavily is a search API optimized for LLMs. Let's add real web search capability!

In [ ]:
import os

# Set Tavily API key
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

# Import Tavily search tool
from langchain_community.tools.tavily_search import TavilySearchResults

# Create search tool
search_tool = TavilySearchResults(
    max_results=3,
    search_depth="basic",
    include_answer=True
)

print(f"Search tool created!")
print(f"Name: {search_tool.name}")
print(f"Description: {search_tool.description[:100]}...")

In [ ]:
# Test the search tool
print("Testing Tavily search...")
result = search_tool.invoke("What are the latest developments in AI agents?")
print(result)

In [ ]:
# Create a research agent with search + calculator
research_tools = [search_tool, calculator, get_weather]

research_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research assistant with access to:
- Web search (tavily_search_results_json) for finding current information
- Calculator for math operations
- Weather lookup for current weather

When answering questions:
1. Search for relevant information when needed
2. Cite your sources when possible
3. Use the calculator for any numerical computations
4. Provide clear, well-structured answers"""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

research_agent = create_tool_calling_agent(llm, research_tools, research_prompt)

research_executor = AgentExecutor(
    agent=research_agent,
    tools=research_tools,
    verbose=True,
    max_iterations=5
)

print("Research agent ready!")

In [ ]:
# Test the research agent
print("=" * 60)
print("RESEARCH QUERY")
print("=" * 60)

result = research_executor.invoke({
    "input": "What is LangChain and what are its main features? Also, what's 42 squared?"
})
print(f"\nFinal Answer:\n{result['output']}")

## Section 2.2: Agent Types Deep Dive

Let's compare **Tool Calling Agent** vs **ReAct Agent**.

In [ ]:
from langchain.agents import create_react_agent
from langchain import hub

# Get the ReAct prompt from LangChain Hub
react_prompt = hub.pull("hwchase17/react")

print("ReAct Prompt Template:")
print(react_prompt.template[:500])
print("...")

In [ ]:
# Create ReAct agent
react_agent = create_react_agent(llm, [calculator, get_weather], react_prompt)

react_executor = AgentExecutor(
    agent=react_agent,
    tools=[calculator, get_weather],
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

print("ReAct agent ready!")

In [ ]:
# Compare: ReAct agent shows explicit reasoning
print("=" * 60)
print("REACT AGENT - Notice the Thought/Action/Observation pattern")
print("=" * 60)

result = react_executor.invoke({
    "input": "What's the weather in Berlin and what's 100 divided by 4?"
})
print(f"\nFinal Answer: {result['output']}")

### Agent Type Comparison

| Feature | Tool Calling Agent | ReAct Agent |
|---------|-------------------|-------------|
| Reasoning | Implicit | Explicit (Thought/Action/Observation) |
| Tokens | Fewer | More (reasoning text) |
| Debugging | Harder | Easier (can see thoughts) |
| Model Requirement | Function calling support | Any chat model |
| Reliability | Higher | Medium |

## Section 2.3: Custom Tools with Pydantic Schemas

For complex tools, use Pydantic schemas for structured inputs.

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import Optional

# Define input schema with Pydantic
class TemperatureConversionInput(BaseModel):
    """Input for temperature conversion."""
    temperature: float = Field(description="The temperature value to convert")
    from_unit: str = Field(description="Source unit: 'celsius' or 'fahrenheit'")
    to_unit: str = Field(description="Target unit: 'celsius' or 'fahrenheit'")

def convert_temperature(temperature: float, from_unit: str, to_unit: str) -> str:
    """Convert temperature between Celsius and Fahrenheit."""
    from_unit = from_unit.lower()
    to_unit = to_unit.lower()
    
    if from_unit == to_unit:
        return f"{temperature}°{from_unit[0].upper()} (no conversion needed)"
    
    if from_unit == "celsius" and to_unit == "fahrenheit":
        result = (temperature * 9/5) + 32
        return f"{temperature}°C = {result:.1f}°F"
    elif from_unit == "fahrenheit" and to_unit == "celsius":
        result = (temperature - 32) * 5/9
        return f"{temperature}°F = {result:.1f}°C"
    else:
        return "Invalid units. Use 'celsius' or 'fahrenheit'"

# Create the tool with schema
temp_converter = StructuredTool.from_function(
    func=convert_temperature,
    name="temperature_converter",
    description="Convert temperature between Celsius and Fahrenheit. Use when you need to convert temperature values between units.",
    args_schema=TemperatureConversionInput
)

print("Structured tool created!")
print(f"Args schema: {temp_converter.args_schema.schema()}")

In [ ]:
# Test the structured tool
print(temp_converter.invoke({"temperature": 25, "from_unit": "celsius", "to_unit": "fahrenheit"}))
print(temp_converter.invoke({"temperature": 98.6, "from_unit": "fahrenheit", "to_unit": "celsius"}))

In [ ]:
# Use in an agent
converter_tools = [get_weather, temp_converter]

converter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a weather assistant. You can:
1. Get weather for any city using get_weather
2. Convert temperatures using temperature_converter

When asked about weather in different units, get the weather first,
then convert the temperature."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

converter_agent = create_tool_calling_agent(llm, converter_tools, converter_prompt)
converter_executor = AgentExecutor(
    agent=converter_agent,
    tools=converter_tools,
    verbose=True
)

result = converter_executor.invoke({
    "input": "What's the weather in Sydney? Give me the temperature in both Celsius and Fahrenheit."
})
print(f"\nAnswer: {result['output']}")

## Section 2.4: Advanced Guardrails

Protecting against runaway agents and handling errors.

In [ ]:
# Create an agent with all guardrails
safe_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    
    # Guardrail 1: Limit iterations
    max_iterations=3,
    
    # Guardrail 2: Time limit (in seconds)
    max_execution_time=30,
    
    # Guardrail 3: Handle parsing errors gracefully
    handle_parsing_errors=True,
    
    # Guardrail 4: What to do when max iterations reached
    early_stopping_method="generate"  # Options: "force" or "generate"
)

print("Safe agent with guardrails ready!")
print(f"Max iterations: 3")
print(f"Max execution time: 30 seconds")
print(f"Handle parsing errors: True")

In [ ]:
# Test with a complex query that might loop
result = safe_executor.invoke({
    "input": "Check weather in 5 different cities around the world"
})
print(f"\nResult: {result['output']}")

In [ ]:
# Demonstrate error handling with a deliberately tricky tool

@tool
def unreliable_tool(query: str) -> str:
    """A tool that sometimes fails. Use for testing error handling."""
    import random
    if random.random() < 0.5:
        raise Exception("Random failure occurred!")
    return f"Success: processed '{query}'"

# Agent with error-prone tool
error_test_executor = AgentExecutor(
    agent=create_tool_calling_agent(llm, [unreliable_tool], prompt),
    tools=[unreliable_tool],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3
)

print("Testing error handling...")
try:
    result = error_test_executor.invoke({"input": "Test the unreliable tool"})
    print(f"Result: {result['output']}")
except Exception as e:
    print(f"Agent handled the error: {e}")

## Section 2.5: Practical Project - Research Assistant

Let's build a complete research assistant that combines everything we've learned!

In [ ]:
# Complete Research Assistant with all features

# Collect all our tools
all_tools = [
    search_tool,      # Web search
    calculator,       # Math
    get_weather,      # Weather
    temp_converter,   # Temperature conversion
]

# Create comprehensive prompt
research_assistant_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intelligent research assistant with access to multiple tools:

TOOLS AVAILABLE:
1. tavily_search_results_json - Search the web for current information
2. calculator - Perform mathematical calculations
3. get_weather - Get current weather for any city
4. temperature_converter - Convert between Celsius and Fahrenheit

GUIDELINES:
- Break complex questions into steps
- Use the appropriate tool for each subtask
- Always verify calculations with the calculator
- Cite sources when using web search
- Provide clear, well-structured answers
- If you're unsure, say so rather than guessing

Remember: You have memory of our conversation, so reference previous context when relevant."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Create memory for multi-turn conversations
research_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create the agent
research_assistant_agent = create_tool_calling_agent(llm, all_tools, research_assistant_prompt)

# Create executor with all guardrails
research_assistant = AgentExecutor(
    agent=research_assistant_agent,
    tools=all_tools,
    memory=research_memory,
    verbose=True,
    max_iterations=10,
    max_execution_time=60,
    handle_parsing_errors=True
)

print("Research Assistant ready!")
print(f"Tools: {[t.name for t in all_tools]}")

In [ ]:
# Interactive research session

print("=" * 60)
print("RESEARCH SESSION - Turn 1")
print("=" * 60)

result1 = research_assistant.invoke({
    "input": "What's the current weather in San Francisco and what are the top tech companies headquartered there?"
})
print(f"\n{result1['output']}")

In [ ]:
print("=" * 60)
print("RESEARCH SESSION - Turn 2 (Follow-up)")
print("=" * 60)

result2 = research_assistant.invoke({
    "input": "Convert that temperature to Fahrenheit. Also, what's the combined market cap of the top 2 companies you mentioned?"
})
print(f"\n{result2['output']}")

In [ ]:
print("=" * 60)
print("RESEARCH SESSION - Turn 3 (Complex query)")
print("=" * 60)

result3 = research_assistant.invoke({
    "input": "Compare the weather in Tokyo vs the city we discussed earlier. Which is warmer right now?"
})
print(f"\n{result3['output']}")

## Summary: What You've Learned

### Part 1 - Core Concepts:
- **Chains vs Agents**: Chains = fixed path, Agents = dynamic decisions
- **Tools**: Functions agents can call (name, description, schema matter!)
- **Agent Loop**: Goal → Think → Act → Observe → Repeat
- **Memory**: Enables multi-turn conversations
- **Debugging**: Always use verbose=True

### Part 2 - Advanced Topics:
- **Tavily Search**: Real web search for current information
- **Agent Types**: Tool Calling (reliable) vs ReAct (transparent)
- **Pydantic Schemas**: Complex structured tool inputs
- **Guardrails**: max_iterations, timeouts, error handling
- **Building Real Apps**: Combining multiple tools with memory

### Best Practices:
1. Start simple, add complexity gradually
2. Write clear tool descriptions - they're crucial!
3. Always set max_iterations and handle_parsing_errors
4. Use verbose=True during development
5. Test edge cases before production

In [ ]:
# Your turn! Try modifying and experimenting:

# 1. Create a new custom tool
# 2. Add it to the research assistant
# 3. Test with different queries

# Example: Create a tool that generates a random fact
# @tool
# def your_custom_tool(param: str) -> str:
#     """Description of what your tool does."""
#     # Your implementation
#     return result

print("Workshop complete! Now it's your turn to experiment.")